In [1]:
#!/usr/bin/env python
# coding: utf-8


import os
#Directory here
os.chdir('/Users/cherryleheu/Codes/NIDIS-Codes/H-RIP/Python')


import geopandas as gpd
import shapely 
from shapely.geometry import Point
import matplotlib.pyplot as plt
# from dms2dec.dms_convert import dms2dec
import numpy as np
import pandas as pd
import rasterio
# from rasterio.plot import show
from rasterstats import zonal_stats
from datetime import datetime, timedelta
import calendar
from dateutil.relativedelta import *
from standard_precip import spi

/opt/anaconda3/envs/hrip/lib/python3.14/site-packages/pyproj/__init__.py:96: UserWarning: Valid PROJ data directory not found. Either set the path using the environmental variable PROJ_DATA (PROJ 9.1+) | PROJ_LIB (PROJ<9.1) or with `pyproj.datadir.set_data_dir`.
  warnings.warn(str(err))


In [5]:
# Read the shapefile
gdf = gpd.read_file('./shapefiles/RID.shp')
thisYear=(datetime.today().strftime("%Y"))
thisMonth=int((datetime.today().strftime("%m")))
# yestYr, yestMonth, yestDay = int((datetime.today() + relativedelta(days=-1)).strftime("%Y")),int((datetime.today() + relativedelta(days=-1)).strftime("%m")),int((datetime.today() + relativedelta(days=-1)).strftime("%d"))
months = np.arange(1,13,1)

for index, row in gdf.iterrows():
    ranchid = row['Polygon']
    geom = row.geometry
    temp = pd.DataFrame({'Year': [],'Month':[],'tmean_f': []})
    for i in np.arange(1990,int(thisYear)+1):
        for j in months:
            if int(i) == int(thisYear) and int(j)==(thisMonth):
                break
            with rasterio.open(f'/Users/cherryleheu/Documents/HCDP/Data/monthly/tmean/tmean_{i}_{j:02d}.tif') as src:
                affine = src.transform
                array = src.read(1)
                df_zonal_stats = pd.DataFrame(zonal_stats(geom, array, affine=affine,nodata=-9999,stats = ['mean']))
            temp_f = 9/5*df_zonal_stats['mean'].iloc[0]+32
            new_row = pd.DataFrame({'Year':i,'Month':j,'tmean_f':temp_f},index=[0])
            temp=pd.concat([temp, new_row],ignore_index=True)
    temp['datetime']=pd.date_range('1/1/1990','1/1/2026',freq='MS')
    temp.to_csv(f'../RID/{ranchid}/{ranchid}_tmean.csv',index=False)

In [8]:
gdf = gpd.read_file('./shapefiles/RID.shp')
lastMonth = int((datetime.today() + relativedelta(months=-1)).strftime("%m"))
lastMonthYr = (datetime.today() + relativedelta(months=-1)).strftime("%Y")
lastMonthDays = calendar.monthrange(int(lastMonthYr), int(lastMonth))[1]

thisYear=(datetime.today().strftime("%Y"))
thisMonth=int((datetime.today().strftime("%m")))

for index, row in gdf.iterrows():
    ranchid = row['Polygon']
    geom = row['geometry']
    lastMonthDailyRF = []
    thisMonthDailyRF = []
    pd.options.display.float_format = '{:.6f}'.format
    for day in np.arange(1,lastMonthDays+1):
        with rasterio.open(f'./rain_daily_maps/{thisYear}_{lastMonth:02d}_{day:02d}.tif') as src:
            affine = src.transform
            array = src.read(1)
            df_zonal_stats = pd.DataFrame(zonal_stats(geom, array, affine=affine,nodata=src.nodata,stats = ['mean']))
        rf = (df_zonal_stats['mean'].iloc[0])/25.4
        row = {'Year': lastMonthYr, 'Month': lastMonth, 'Day':day, 'RF_in': rf}
        lastMonthDailyRF.append(row)
    df = pd.DataFrame(lastMonthDailyRF)
    df.to_csv(f'../RID/{ranchid}/{ranchid}_rf_daily_last_month.csv',index=False)